# SQLite 基础使用示例

这个 notebook 演示如何用 Python 连接 `data/processed/course_reviews_simple.sqlite`，并使用基础 SQL 查询课程评论和当前学期教学班数据。

数据库由以下脚本生成和更新：

- `src/build_simple_reviews_sqlite.py`：生成历史评论明细和评分汇总表
- `src/ingest_course_plus.py`：导入 Course+ 当前学期教学班表

主要包含三张表：

- `course_teacher_reviews`：评论明细表
- `course_teacher_rating_summary`：课程-教师评分汇总表
- `course_plus_offerings`：当前学期教学班信息表

## 1. 连接 SQLite 数据库

SQLite 是一个文件型数据库，不需要启动数据库服务，只需要连接 `.sqlite` 文件。

In [16]:
import sqlite3
from pathlib import Path

DB_PATH = Path('../data/processed/course_reviews_simple.sqlite')
connection = sqlite3.connect(DB_PATH)

DB_PATH.exists(), DB_PATH

(True, WindowsPath('../data/processed/course_reviews_simple.sqlite'))

## 2. 查看数据库里有哪些表

SQLite 的元数据保存在 `sqlite_master` 表中。

In [17]:
cursor = connection.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""")

cursor.fetchall()

[('course_plus_offerings',),
 ('course_teacher_rating_summary',),
 ('course_teacher_reviews',)]

## 3. 查看表结构

`PRAGMA table_info(表名)` 可以查看字段名、类型等信息。

In [18]:
connection.execute("PRAGMA table_info(course_teacher_reviews);").fetchall()

[(0, 'review_id', 'TEXT', 0, None, 0),
 (1, 'course_id', 'TEXT', 0, None, 0),
 (2, 'course_code', 'TEXT', 0, None, 0),
 (3, 'course_name', 'TEXT', 0, None, 0),
 (4, 'course_teacher', 'TEXT', 0, None, 0),
 (5, 'rating', 'REAL', 0, None, 0),
 (6, 'score', 'TEXT', 0, None, 0),
 (7, 'semester_name', 'TEXT', 0, None, 0),
 (8, 'comment', 'TEXT', 0, None, 0),
 (9, 'comment_length', 'TEXT', 0, None, 0),
 (10, 'created_at', 'TEXT', 0, None, 0),
 (11, 'modified_at', 'TEXT', 0, None, 0)]

## 4. 查询前几条评论

`SELECT` 用于选择字段，`LIMIT` 用于限制返回行数。

In [19]:
connection.execute("""
SELECT review_id, course_code, course_name, course_teacher, rating, semester_name
FROM course_teacher_reviews
LIMIT 5;
""").fetchall()

[('60444', 'JCCX0004', '医疗机器人的机构与控制', '陈卫东', 5.0, '2025-2026-2'),
 ('60443', 'PE3204', '工程流体力学（B类）', '彭迪', 5.0, '2025-2026-1'),
 ('39800', 'MECH2508H', '理论力学（荣誉）', '刘锦阳', 4.0, '2024-2025-1'),
 ('60442', 'ME3604', '汽车构造与设计', '朱平', 1.0, '2025-2026-2'),
 ('60385', 'MATH1203', '数学分析I', '梁进', 5.0, '2025-2026-1')]

## 5. 按课程号和教师名筛选评论

`WHERE` 用于设置筛选条件。这里查询课程 `JCCX0004`、教师 `陈卫东` 的评论。

In [20]:
connection.execute("""
SELECT review_id, course_code, course_name, course_teacher, rating, semester_name, comment
FROM course_teacher_reviews
WHERE course_code = 'JCCX0004'
  AND course_teacher = '陈卫东'
ORDER BY modified_at DESC;
""").fetchall()

[('60444',
  'JCCX0004',
  '医疗机器人的机构与控制',
  '陈卫东',
  5.0,
  '2025-2026-2',
  '课程内容：前半学期介绍医疗机器人的种类和控制原理，比较偏理论，有几次作业。后半学期去转化医学楼做项目。项目是设计医疗机器人的RCM算法，只需要写代码和仿真。这课是陈卫东老师和高安柱老师一起讲的课，所以我在这里一起点评了。\n\n上课自由度：两个老师都不管学生在下面干嘛。后面有几节课来了不到一半的人也没说什么。老师最后一节课还疯狂安利他的PRP项目。\n# 但是\n陈卫东老师上课很有激情，喜欢跑到学生面前来讲，尤其是喜欢围着看起来认真听讲的人转，对于想要开小差的同学压力很大。\n\n考核标准：暂时没说\n\n授课质量：很好，老师讲的很仔细，可惜没人听。后面项目也有助教手把手教。'),
 ('24017',
  'JCCX0004',
  '医疗机器人的机构与控制',
  '陈卫东',
  5.0,
  '2022-2023-2',
  '课程内容：医疗机器人基础知识，主要做项目；之前几乎机器人学0基础，还是学了很多东西的\n\n上课自由度：参考楼上，自由度超高，有事请假什么都不影响成绩\n\n考核标准：一次小作业+一个课程大项目（第二次上课开题）\n\n授课质量：高！！每个项目配备助教！！有事都可以联系老师，老师会很快给一些建议和帮助！！\n\n强烈安利！！！嘶吼！！！'),
 ('23882',
  'JCCX0004',
  '医疗机器人的机构与控制',
  '陈卫东',
  5.0,
  '2022-2023-2',
  '课程内容：运动控制相关知识（一点点），主要内容是是自主做课题项目（一共11次课，做项目的时间超过一半）\n上课自由度：小班授课+会议室圆桌（这学期一共只有不到10人，甚至期末汇报只有3组（6人），大家快来选课，这个宝藏课！），有事情请假也没关系\n考核标准：一次小作业+一个课程大项目（第二次上课开题）\n授课质量：老师讲的详细生动，给项目推进提出许多建议，上课内容也应用在项目中。项目占用时间不多，课下没有用过时间，每次上课时间+课后留下一段时间就完成了项目。每个项目小组会有一个单独的助教指导，助教给我们很多很多的指导帮助，非常nice！这个过程也有许多收获，强烈安利！'),
 ('2266

## 6. 使用参数化查询

不要用字符串拼接构造用户输入的 SQL。参数化查询更安全，也更适合复用。

In [21]:
course_code = 'JCCX0004'
teacher = '陈卫东'

connection.execute("""
SELECT review_id, rating, semester_name, comment
FROM course_teacher_reviews
WHERE course_code = ?
  AND course_teacher = ?
ORDER BY modified_at DESC;
""", (course_code, teacher)).fetchall()

[('60444',
  5.0,
  '2025-2026-2',
  '课程内容：前半学期介绍医疗机器人的种类和控制原理，比较偏理论，有几次作业。后半学期去转化医学楼做项目。项目是设计医疗机器人的RCM算法，只需要写代码和仿真。这课是陈卫东老师和高安柱老师一起讲的课，所以我在这里一起点评了。\n\n上课自由度：两个老师都不管学生在下面干嘛。后面有几节课来了不到一半的人也没说什么。老师最后一节课还疯狂安利他的PRP项目。\n# 但是\n陈卫东老师上课很有激情，喜欢跑到学生面前来讲，尤其是喜欢围着看起来认真听讲的人转，对于想要开小差的同学压力很大。\n\n考核标准：暂时没说\n\n授课质量：很好，老师讲的很仔细，可惜没人听。后面项目也有助教手把手教。'),
 ('24017',
  5.0,
  '2022-2023-2',
  '课程内容：医疗机器人基础知识，主要做项目；之前几乎机器人学0基础，还是学了很多东西的\n\n上课自由度：参考楼上，自由度超高，有事请假什么都不影响成绩\n\n考核标准：一次小作业+一个课程大项目（第二次上课开题）\n\n授课质量：高！！每个项目配备助教！！有事都可以联系老师，老师会很快给一些建议和帮助！！\n\n强烈安利！！！嘶吼！！！'),
 ('23882',
  5.0,
  '2022-2023-2',
  '课程内容：运动控制相关知识（一点点），主要内容是是自主做课题项目（一共11次课，做项目的时间超过一半）\n上课自由度：小班授课+会议室圆桌（这学期一共只有不到10人，甚至期末汇报只有3组（6人），大家快来选课，这个宝藏课！），有事情请假也没关系\n考核标准：一次小作业+一个课程大项目（第二次上课开题）\n授课质量：老师讲的详细生动，给项目推进提出许多建议，上课内容也应用在项目中。项目占用时间不多，课下没有用过时间，每次上课时间+课后留下一段时间就完成了项目。每个项目小组会有一个单独的助教指导，助教给我们很多很多的指导帮助，非常nice！这个过程也有许多收获，强烈安利！'),
 ('22668', 5.0, '2022-2023-2', '助教的布里茨玩的很好')]

## 7. 查询评分汇总表

第二张表已经提前按 `course_code + course_teacher` 聚合好了评论数量和平均评分。

In [22]:
connection.execute("""
SELECT course_code, course_teacher, course_name, review_count, avg_rating
FROM course_teacher_rating_summary
WHERE course_code = ?
  AND course_teacher = ?;
""", (course_code, teacher)).fetchall()

[('JCCX0004', '陈卫东', '医疗机器人的机构与控制', 4, 5.0)]

## 8. 排序：找评论数最多的课程-教师组合

`ORDER BY` 可以排序，`DESC` 表示降序。

In [23]:
connection.execute("""
SELECT course_code, course_teacher, course_name, review_count, avg_rating
FROM course_teacher_rating_summary
ORDER BY review_count DESC
LIMIT 10;
""").fetchall()

[('EE0502', '张峰', '电路实验', 674, 1.119),
 ('MIL1201', '闫成', '军事理论', 305, 4.898),
 ('MATH1205', '崔振', '线性代数', 217, 3.945),
 ('PHY1221', '王宇兴', '大学物理实验（1）', 194, 2.33),
 ('MATH1201', '陈春丽', '高等数学I', 188, 4.973),
 ('MATH1205', '蒋启芬', '线性代数', 184, 4.701),
 ('PHY1222', '王宇兴', '大学物理实验（2）', 163, 2.067),
 ('MATH1205', '麻志浩', '线性代数', 163, 3.798),
 ('MATH1207', '仇璘', '概率统计', 158, 4.032),
 ('MIL1201', '赵建世', '军事理论', 130, 4.285)]

## 9. 聚合：用 SQL 现场计算平均评分

也可以不依赖汇总表，直接在评论明细表中使用 `COUNT` 和 `AVG`。

In [24]:
connection.execute("""
SELECT
  course_code,
  course_teacher,
  COUNT(*) AS review_count,
  ROUND(AVG(rating), 3) AS avg_rating
FROM course_teacher_reviews
WHERE course_code = ?
  AND course_teacher = ?
GROUP BY course_code, course_teacher;
""", (course_code, teacher)).fetchall()

[('JCCX0004', '陈卫东', 4, 5.0)]

## 10. 模糊搜索课程名

`LIKE` 可以做简单的模糊匹配，`%` 表示任意长度的字符。

In [32]:
keyword = '%工程实践%'

connection.execute("""
SELECT course_code, course_teacher, course_name, review_count, avg_rating
FROM course_teacher_rating_summary
WHERE course_name LIKE ?
ORDER BY review_count DESC
LIMIT 10;
""", (keyword,)).fetchall()

[('SI1210', '冷春涛', '工程实践', 70, 2.586),
 ('EE1503', '刘彦博', '工程实践与科技创新I', 64, 1.625),
 ('SI1210', '陶波', '工程实践', 58, 2.586),
 ('ICE2502', '袁焱', '工程实践与科技创新Ⅲ-A', 48, 2.479),
 ('SI1213', '冷春涛', '工程实践（C类）', 24, 2.25),
 ('AU2514', '王冰', '工程实践与科技创新Ⅳ-D', 21, 4.19),
 ('EE0503', '何哲陟', '工程实践与科技创新II', 15, 1.267),
 ('EE1503', '陈颖琪', '工程实践与科技创新I', 15, 3.133),
 ('CS3512', '周煊赫', '工程实践与科技创新Ⅲ-E', 14, 1.071),
 ('ICE0501', '殳国华', '工程实践与科技创新Ⅱ-A', 14, 1.357)]

## 11. 用 pandas 显示查询结果

如果安装了 pandas，可以用 `read_sql_query` 把 SQL 结果显示成表格。

In [26]:
try:
    import pandas as pd

    df = pd.read_sql_query("""
    SELECT course_code, course_teacher, course_name, review_count, avg_rating
    FROM course_teacher_rating_summary
    ORDER BY review_count DESC
    LIMIT 10;
    """, connection)
    display(df)
except ImportError:
    print('当前环境未安装 pandas，可以跳过这一节。')

,course_code,course_teacher,course_name,review_count,avg_rating
0,EE0502,张峰,电路实验,674,1.119
1,MIL1201,闫成,军事理论,305,4.898
2,MATH1205,崔振,线性代数,217,3.945
3,PHY1221,王宇兴,大学物理实验（1）,194,2.330
4,MATH1201,陈春丽,高等数学I,188,4.973
5,MATH1205,蒋启芬,线性代数,184,4.701
6,PHY1222,王宇兴,大学物理实验（2）,163,2.067
7,MATH1205,麻志浩,线性代数,163,3.798
8,MATH1207,仇璘,概率统计,158,4.032
9,MIL1201,赵建世,军事理论,130,4.285


## 12. 查找开课信息



In [33]:
course_code = 'SI1210'

connection.execute("""
SELECT
  course_code,
  course_name,
  teacher,
  teaching_class,
  campus,
  course_type,
  enrollment,
  class_capacity,
  schedule_text,
  schedule_code
FROM course_plus_offerings
WHERE course_code = ?
ORDER BY teaching_class;
""", (course_code,)).fetchall()

[('SI1210',
  '工程实践',
  '冷春涛',
  '(2025-2026-1)-SI1210-01',
  '闵行',
  '必修',
  230,
  231.0,
  '星期一第5-10节{1-16周}',
  'D1:P5-10:W1-16'),
 ('SI1210',
  '工程实践',
  '冷春涛',
  '(2025-2026-1)-SI1210-02',
  '闵行',
  '必修',
  281,
  281.0,
  '星期二第5-10节{1周};星期二第5-10节{2-16周}',
  'D2:P5-10:W1;D2:P5-10:W2-16'),
 ('SI1210',
  '工程实践',
  '冷春涛',
  '(2025-2026-1)-SI1210-03',
  '闵行',
  '必修',
  288,
  288.0,
  '星期四第5-10节{1周};星期四第5-10节{2-16周}',
  'D4:P5-10:W1;D4:P5-10:W2-16'),
 ('SI1210',
  '工程实践',
  '冷春涛',
  '(2025-2026-1)-SI1210-04',
  '闵行',
  '必修',
  213,
  214.0,
  '星期五第5-10节{1-16周}',
  'D5:P5-10:W1-16')]

## 13. 批量查找开课信息


In [30]:
course_codes = [
    'ECON1501',
    'SI1214',
    'BUSS2515',
    'PE003C28',
    'BUSS2302',
    'ECON2404',
    'ECON2403'
]

values_clause = ', '.join(['(?, ?)'] * len(course_codes))
params = [value for item in enumerate(course_codes) for value in (item[1], item[0])]

sql = f"""
WITH requested_courses(course_code, input_order) AS (
  VALUES {values_clause}
)
SELECT
  requested_courses.course_code AS requested_course_code,
  course_plus_offerings.course_code,
  course_plus_offerings.course_name,
  course_plus_offerings.teacher,
  course_plus_offerings.teaching_class,
  course_plus_offerings.campus,
  course_plus_offerings.course_type,
  course_plus_offerings.enrollment,
  course_plus_offerings.class_capacity,
  course_plus_offerings.schedule_text,
  course_plus_offerings.schedule_code
FROM requested_courses
LEFT JOIN course_plus_offerings
  ON course_plus_offerings.course_code = requested_courses.course_code
ORDER BY requested_courses.input_order, course_plus_offerings.teaching_class;
"""

try:
    import pandas as pd

    offerings_df = pd.read_sql_query(sql, connection, params=params)
    display(offerings_df)
except ImportError:
    connection.execute(sql, params).fetchall()

,requested_course_code,course_code,course_name,teacher,teaching_class,campus,course_type,enrollment,class_capacity,schedule_text,schedule_code
0,ECON1501,ECON1501,金融科技概论,张澄宇,(2025-2026-1)-ECON1501-01,闵行,"必修,限选",47.0,50.0,星期二第1-2节{1-16周},D2:P1-2:W1-16
1,ECON1501,ECON1501,金融科技概论,张澄宇,(2025-2026-1)-ECON1501-02,闵行,"必修,限选",39.0,40.0,星期二第3-4节{1-16周},D2:P3-4:W1-16
2,SI1214,None,None,None,None,None,None,NaN,NaN,None,None
3,BUSS2515,BUSS2515,会计学,张书博,(2025-2026-1)-BUSS2515-01,闵行,限选,35.0,35.0,星期四第6-8节{1-16周},D4:P6-8:W1-16
4,BUSS2515,BUSS2515,会计学,丁慧,(2025-2026-1)-BUSS2515-02,闵行,限选,35.0,35.0,星期四第6-8节{1-16周},D4:P6-8:W1-16
5,BUSS2515,BUSS2515,会计学,黄立,(2025-2026-1)-BUSS2515-03,闵行,限选,35.0,35.0,星期四第6-8节{1-16周},D4:P6-8:W1-16
6,BUSS2515,BUSS2515,会计学,郝盛泉,(2025-2026-1)-BUSS2515-04,闵行,限选,35.0,35.0,星期四第6-8节{1-16周},D4:P6-8:W1-16
7,BUSS2515,BUSS2515,会计学,石桂峰,(2025-2026-1)-BUSS2515-05,闵行,限选,35.0,35.0,星期四第6-8节{1-16周},D4:P6-8:W1-16
8,BUSS2515,BUSS2515,会计学,王惠,(2025-2026-1)-BUSS2515-06,闵行,限选,40.0,25.0,星期三第11-13节{1-16周},D3:P11-13:W1-16
9,PE003C28,PE003C28,舞龙,顾剑平,(2025-2026-1)-PE003C28-01,闵行,必修,30.0,28.0,星期二第7-8节{1-16周},D2:P7-8:W1-16


## 13. 理解 `schedule_code` 排课编码

`schedule_text` 中类似 `星期一第1-2节{1-8周}` 的文本会被转成更方便程序处理的编码：

- `D1` 到 `D7`：星期一到星期日
- `P1-2`：第 1-2 节
- `W1-8`：第 1-8 周
- 多个时间段用 `;` 分隔
- 同一天多个节次或周次用 `+` 分隔

例如：

`星期一第1-2节{1-8周};星期五第1-2节{1-8周}`

会变成：

`D1:P1-2:W1-8;D5:P1-2:W1-8`

In [14]:
connection.execute("""
SELECT course_code, course_name, teacher, teaching_class, schedule_code
FROM course_plus_offerings
WHERE schedule_code LIKE '%D1:P1-2:%'
ORDER BY course_code
LIMIT 10;
""").fetchall()

[('AE2201',
  '工程力学：静力学与动力学',
  '范寅',
  '(2025-2026-1)-AE2201-01',
  'D1:P1-2:W1-16;D4:P9-10:W1-16'),
 ('AI1806',
  '信号与系统（B类）',
  '陈思衡',
  '(2025-2026-1)-AI1806-01',
  'D1:P1-2:W1-16;D4:P3-4:W1-15周(单)'),
 ('BIO1254', '普通生物学', '陶美凤', '(2025-2026-1)-BIO1254-03', 'D1:P1-2:W1-16'),
 ('BIO1254', '普通生物学', '杨立桃', '(2025-2026-1)-BIO1254-04', 'D1:P1-2:W1-16'),
 ('BIO1254', '普通生物学', '唐满成', '(2025-2026-1)-BIO1254-07', 'D1:P1-2:W1-16'),
 ('BIO1254', '普通生物学', '霍云龙', '(2025-2026-1)-BIO1254-01', 'D1:P1-2:W1-16'),
 ('BIO1254', '普通生物学', '李雷', '(2025-2026-1)-BIO1254-02', 'D1:P1-2:W1-16'),
 ('BIO1254', '普通生物学', '顾若虚', '(2025-2026-1)-BIO1254-05', 'D1:P1-2:W1-16'),
 ('BIO2231',
  '生物化学',
  '林双君',
  '(2025-2026-1)-BIO2231-01',
  'D1:P1-2:W1-16;D4:P1-2:W1-16'),
 ('BIO3504', '应用生物信息学', '欧竑宇', '(2025-2026-1)-BIO3504-01', 'D1:P1-2:W1-16')]

## 12. 关闭数据库连接

使用完数据库后关闭连接。

In [15]:
connection.close()